In [ ]:
import os
from pyspark.sql import SparkSession

warehouse_path = "/tmp/ehdip_warehouse"
spark = SparkSession.builder \
    .appName("Day07_Upsert") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_silver")

# Create target stateful table
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_silver.omop_person (
    person_id BIGINT,
    gender_concept_id INT,
    person_source_value STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.ehdip_silver.omop_person
VALUES (101, 8507, 'MRN-A', timestamp('2023-01-01 10:00:00')),
       (102, 8532, 'MRN-B', timestamp('2023-01-01 10:00:00'))
""")

print("INITIAL TARGET TABLE STATE:")
spark.sql("SELECT * FROM local.ehdip_silver.omop_person ORDER BY person_id").show()

# Simulate incoming CDC micro-batch (Updates and Deletes)
cdc_updates = [
    (101, 8507, 'MRN-A-UPDATED', '2023-01-02 12:00:00', 'u'), # Update (newer timestamp)
    (102, 8532, 'MRN-B-OLD', '2022-12-31 08:00:00', 'u'), # Out-of-order update (should be ignored)
    (103, 0, 'MRN-C-NEW', '2023-01-02 12:00:00', 'c'), # New Insert
    (101, 8507, 'MRN-A-DEL', '2023-01-03 00:00:00', 'd') # Delete operation (if processed after update)
]
cdc_df = spark.createDataFrame(cdc_updates, ["person_id", "gender_concept_id", "person_source_value", "updated_at", "op"])
cdc_df = cdc_df.withColumn("updated_at", cdc_df.updated_at.cast("timestamp"))
cdc_df.createOrReplaceTempView("cdc_microbatch")

print("INCOMING CDC MICRO-BATCH:")
spark.sql("SELECT * FROM cdc_microbatch").show()

# Apply MERGE INTO logic (excluding op from final table)
spark.sql("""
MERGE INTO local.ehdip_silver.omop_person t
USING cdc_microbatch s
ON t.person_id = s.person_id
WHEN MATCHED AND s.op = 'd' THEN 
    DELETE
WHEN MATCHED AND s.updated_at > t.updated_at THEN 
    UPDATE SET 
        t.gender_concept_id = s.gender_concept_id,
        t.person_source_value = s.person_source_value,
        t.updated_at = s.updated_at
WHEN NOT MATCHED AND s.op != 'd' THEN 
    INSERT (person_id, gender_concept_id, person_source_value, updated_at)
    VALUES (s.person_id, s.gender_concept_id, s.person_source_value, s.updated_at)
""")

print("TARGET TABLE STATE AFTER MERGE:")
spark.sql("SELECT * FROM local.ehdip_silver.omop_person ORDER BY person_id").show()
